## System Tables — Already Enabled

Your serverless workspace already has system tables enabled and populated. The `system` catalog contains 11 schemas with operational data about your account — billing, audit logs, query history, job runs, lineage, and more.

This notebook explores what's available and runs sample queries across key system tables.

In [0]:
%sql
-- List all available system schemas
SHOW SCHEMAS IN system

In [0]:
%sql
-- Billing usage summary by SKU and product
SELECT
  billing_origin_product,
  sku_name,
  usage_date,
  ROUND(SUM(usage_quantity), 2) AS total_dbus
FROM system.billing.usage
GROUP BY billing_origin_product, sku_name, usage_date
ORDER BY usage_date DESC, total_dbus DESC

In [0]:
%sql
-- Total cost = DBUs × effective list price per DBU (USD)
SELECT
  u.billing_origin_product,
  u.sku_name,
  u.usage_date,
  ROUND(SUM(u.usage_quantity), 2) AS total_dbus,
  ROUND(p.pricing.effective_list.default, 4) AS price_per_dbu,
  ROUND(SUM(u.usage_quantity) * p.pricing.effective_list.default, 2) AS total_cost_usd
FROM system.billing.usage u
JOIN system.billing.list_prices p
  ON u.sku_name = p.sku_name
  AND u.cloud = p.cloud
  AND u.usage_start_time >= p.price_start_time
  AND (p.price_end_time IS NULL OR u.usage_start_time < p.price_end_time)
GROUP BY u.billing_origin_product, u.sku_name, u.usage_date, p.pricing.effective_list.default
ORDER BY u.usage_date DESC, total_cost_usd DESC

In [0]:
%sql
-- Recent query history
SELECT
  statement_id,
  executed_by,
  statement_type,
  status,
  total_duration_ms,
  execution_start_time_utc,
  statement_text
FROM system.query.history
ORDER BY execution_start_time_utc DESC
LIMIT 20

In [0]:
%sql
-- List all tables across all system schemas
SELECT table_schema, table_name, comment
FROM system.information_schema.tables
WHERE table_catalog = 'system'
ORDER BY table_schema, table_name

In [0]:
%sql
-- Audit log sample — recent account activity
SELECT
  event_time,
  service_name,
  action_name,
  user_identity.email AS user_email,
  source_ip_address
FROM system.access.audit
ORDER BY event_time DESC
LIMIT 20

## Available System Schemas

| Schema | Contains |
| --- | --- |
| `system.access` | Audit logs, table/column lineage, workspace info |
| `system.ai` | AI feature usage |
| `system.ai_gateway` | AI Gateway endpoint usage |
| `system.billing` | Billable usage (DBUs) by SKU, product, workspace |
| `system.compute` | Cluster and warehouse configurations/events |
| `system.lakeflow` | Jobs, job runs, and task timelines |
| `system.mlflow` | MLflow experiment and model tracking |
| `system.query` | Query execution history |
| `system.serving` | Model serving endpoint usage |
| `system.storage` | Predictive optimization and storage metrics |